In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# مدخلات ومخرجات Sampler

<Accordion>
<AccordionItem title="إصدارات الحزم">

الكود في هذه الصفحة تم تطويره باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
تقدم هذه الصفحة نظرة عامة على مدخلات ومخرجات الأداة البدائية Sampler في Qiskit Runtime، التي تنفذ أعمال التشغيل على موارد الحوسبة IBM Quantum&reg;. يتيح لك Sampler تعريف أعمال تشغيل موجّهة بكفاءة باستخدام بنية بيانات تُعرف بـ [**Primitive Unified Bloc (PUB)**](/guides/primitive-input-output#pubs). يتم استخدامها كمدخلات لطريقة [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2#run) للأداة البدائية Sampler، التي تنفذ أعمال التشغيل المحددة كمهمة. ثم، بعد اكتمال المهمة، يتم إرجاع النتائج بتنسيق يعتمد على كل من PUBs المستخدمة وخيارات وقت التشغيل المحددة من الأداة البدائية.
## المدخلات
كل PUB بالتنسيق التالي:

(`<دائرة واحدة>`، `<قيمة أو أكثر من قيم المعاملات الاختيارية>`، `<shots اختيارية>`)،

يمكن أن تكون هناك عناصر متعددة من `قيم المعاملات`، ويمكن أن يكون كل عنصر إما مصفوفة أو معاملاً واحداً، اعتماداً على الدائرة المختارة. بالإضافة إلى ذلك، يجب أن تحتوي المدخلات على قياسات.

بالنسبة للأداة البدائية Sampler، يمكن أن يحتوي PUB على ثلاثة قيم كحد أقصى:
- `QuantumCircuit` واحد، والذي قد يحتوي على كائن أو أكثر من كائنات [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
   *ملاحظة: يجب أن تتضمن هذه الدوائر أيضاً تعليمات قياس لكل من القبتات المراد أخذ عيناتها.*
- مجموعة من قيم المعاملات لربط الدائرة بها $\theta_k$ (مطلوبة فقط إذا كانت هناك كائنات `Parameter` يجب ربطها في وقت التشغيل)
- (اختياري) عدد من الـ shots لقياس الدائرة به
---

يوضح الكود التالي مجموعة مثال من المدخلات الموجّهة للأداة البدائية `Sampler` وتنفيذها على Backend IBM&reg; كـ `RuntimeJobV2` واحد.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    SamplerV2 as Sampler,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Sampler object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = Sampler(mode=backend)
job = sampler.run([sampler_pub])
result = job.result()

## المخرجات
بعد إرسال PUB أو أكثر إلى QPU للتنفيذ واكتمال مهمة بنجاح، يتم إرجاع البيانات كـ [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) وهو كائن حاوية يتم الوصول إليه عن طريق استدعاء طريقة `RuntimeJobV2.result()`. يحتوي `PrimitiveResult` على قائمة قابلة للتكرار من كائنات [`SamplerPubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.SamplerPubResult) تحتوي على نتائج التنفيذ لكل PUB. هذه البيانات هي عينات من مخرجات الدائرة.

يتوافق كل عنصر من هذه القائمة مع PUB مُرسَل إلى طريقة `run()` الخاصة بالأداة البدائية (على سبيل المثال، مهمة مُرسَلة مع 20 PUB ستعيد كائن `PrimitiveResult` يحتوي على قائمة من 20 كائن `SamplerPubResult`، يتوافق كل منها مع PUB).

يمتلك كل كائن `SamplerPubResult` سمتي `data` و `metadata`.
    - سمة `data` هي [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) مخصصة تحتوي على قيم القياس الفعلية والانحرافات المعيارية وما شابه ذلك. صناديق البيانات هي كائنات شبيهة بالقاموس تحتوي على `BitArray` واحد لكل `ClassicalRegister` في الدائرة.
    - فئة `BitArray` هي حاوية لبيانات الـ shot المرتبة. تخزن سلاسل البت المأخوذة كعينة كبايتات داخل مصفوفة ثنائية الأبعاد. يمر المحور الأيسر لهذه المصفوفة على الـ shots المرتبة، بينما يمر المحور الأيمن على البايتات.
    - تحتوي سمة `metadata` على معلومات حول خيارات وقت التشغيل المستخدمة (موضح لاحقاً في قسم [بيانات نتائج التعريف](#result-metadata) في هذه الصفحة).

فيما يلي مخطط مرئي لبنية بيانات `PrimitiveResult`:

```
    └── PrimitiveResult
        ├── SamplerPubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── NAME_OF_CLASSICAL_REGISTER
        │       │   └── BitArray of count data (default is 'meas')
        |       |
        │       └── NAME_OF_ANOTHER_CLASSICAL_REGISTER
        │           └── BitArray of count data (exists only if more than one
        |                 ClassicalRegister was specified in the circuit)
        ├── SamplerPubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       └── NAME_OF_CLASSICAL_REGISTER
        |           └── BitArray of count data for second pub
        ├── ...
        ├── ...
        └── ...
```

بكل بساطة، تعيد مهمة واحدة كائن [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) وتحتوي على قائمة من كائن أو أكثر من كائنات [`SamplerPubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.SamplerPubResult). ثم تخزن كائنات `SamplerPubResult` هذه بيانات القياس لكل PUB تم إرساله إلى المهمة.

كمثال أول، لنلقِ نظرة على الدائرة ذات عشرة قبتات التالية:

In [2]:
# generate a ten-qubit GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure_all()

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results
sampler = Sampler(mode=backend)
job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains one BitArray
data = result[0].data
print(f"Databin: {data}\n")

# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method
array = data.meas
print(f"BitArray: {array}\n")
print(f"The shape of register `meas` is {data.meas.array.shape}.\n")
print(f"The bytes in register `alpha`, shot by shot:\n{data.meas.array}\n")

Databin: DataBin(meas=BitArray(<shape=(), num_shots=4096, num_bits=10>))

BitArray: BitArray(<shape=(), num_shots=4096, num_bits=10>)

The shape of register `meas` is (4096, 2).

The bytes in register `alpha`, shot by shot:
[[  0   0]
 [  3 255]
 [  0   0]
 ...
 [  3 255]
 [  2 255]
 [  3 255]]



It can sometimes be convenient to convert away from the bytes format in the `BitArray` to bitstrings. The `get_count` method returns a dictionary mapping bitstrings to the number of times that they occurred.

In [3]:
# optionally, convert away from the native BitArray format to a dictionary format
counts = data.meas.get_counts()
print(f"Counts: {counts}")

Counts: {'0000000000': 1649, '1111111111': 1344, '1111111000': 26, '1101111111': 40, '1111110000': 20, '0010000000': 32, '1000000000': 67, '1111110110': 4, '0000011110': 4, '0000000001': 78, '0010100000': 1, '1100000000': 37, '1111111110': 126, '1111110111': 35, '1111011111': 32, '0011111000': 1, '1011110111': 1, '0000011111': 48, '1111000000': 14, '0110000000': 1, '1110111110': 2, '1110011111': 4, '1111100000': 19, '1101111000': 1, '1111111011': 8, '0001011111': 3, '1110000000': 31, '0000000111': 25, '1110000001': 3, '0011111111': 24, '0000100000': 7, '1111111101': 30, '1111101111': 16, '0111111111': 37, '0000011101': 4, '0101111111': 4, '1011111110': 2, '0000000010': 17, '1011111111': 20, '0000100111': 1, '0010000111': 1, '1011010000': 1, '1101101111': 2, '1011110000': 1, '1000000001': 4, '0000001000': 23, '0011111110': 8, '1111111001': 1, '1100111111': 2, '0000011000': 2, '0001111110': 2, '0000111111': 20, '0001111111': 33, '1110111111': 11, '1010000000': 3, '0111011111': 2, '000000

قد يكون من المفيد أحياناً التحويل من تنسيق البايتات في `BitArray` إلى سلاسل البت. تعيد طريقة `get_count` قاموساً يربط سلاسل البت بعدد مرات حدوثها.

In [4]:
# generate a ten-qubit GHZ circuit with two classical registers
circuit = QuantumCircuit(
    qreg := QuantumRegister(10),
    alpha := ClassicalRegister(1, "alpha"),
    beta := ClassicalRegister(9, "beta"),
)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure([0], alpha)
circuit.measure(range(1, 10), beta)

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results
sampler = Sampler(mode=backend)
job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains two BitArrays, one per register, and can be accessed
# as attributes using the registers' names
data = result[0].data
print(f"BitArray for register 'alpha': {data.alpha}")
print(f"BitArray for register 'beta': {data.beta}")

BitArray for register 'alpha': BitArray(<shape=(), num_shots=4096, num_bits=1>)
BitArray for register 'beta': BitArray(<shape=(), num_shots=4096, num_bits=9>)


### Use `BitArray` objects for performant post-processing

Since arrays generally offer better performance compared to dictionaries, it is advisable to perform any post-processing directly on the `BitArray` objects rather than on dictionaries of counts. The `BitArray` class offers a range of methods to perform some common post-processing operations:

In [5]:
print(f"The shape of register `alpha` is {data.alpha.array.shape}.")
print(f"The bytes in register `alpha`, shot by shot:\n{data.alpha.array}\n")

print(f"The shape of register `beta` is {data.beta.array.shape}.")
print(f"The bytes in register `beta`, shot by shot:\n{data.beta.array}\n")

# post-select the bitstrings of `beta` based on having sampled "1" in `alpha`
mask = data.alpha.array == "0b1"
ps_beta = data.beta[mask[:, 0]]
print(f"The shape of `beta` after post-selection is {ps_beta.array.shape}.")
print(f"The bytes in `beta` after post-selection:\n{ps_beta.array}")

# get a slice of `beta` to retrieve the first three bits
beta_sl_bits = data.beta.slice_bits([0, 1, 2])
print(
    f"The shape of `beta` after bit-wise slicing is {beta_sl_bits.array.shape}."
)
print(f"The bytes in `beta` after bit-wise slicing:\n{beta_sl_bits.array}\n")

# get a slice of `beta` to retrieve the bytes of the first five shots
beta_sl_shots = data.beta.slice_shots([0, 1, 2, 3, 4])
print(
    f"The shape of `beta` after shot-wise slicing is {beta_sl_shots.array.shape}."
)
print(
    f"The bytes in `beta` after shot-wise slicing:\n{beta_sl_shots.array}\n"
)

# calculate the expectation value of diagonal operators on `beta`
ops = [SparsePauliOp("ZZZZZZZZZ"), SparsePauliOp("IIIIIIIIZ")]
exp_vals = data.beta.expectation_values(ops)
for o, e in zip(ops, exp_vals):
    print(f"Exp. val. for observable `{o}` is: {e}")

# concatenate the bitstrings in `alpha` and `beta` to "merge" the results of the two
# registers
merged_results = BitArray.concatenate_bits([data.alpha, data.beta])
print(f"\nThe shape of the merged results is {merged_results.array.shape}.")
print(f"The bytes of the merged results:\n{merged_results.array}\n")

The shape of register `alpha` is (4096, 1).
The bytes in register `alpha`, shot by shot:
[[0]
 [0]
 [0]
 ...
 [0]
 [0]
 [0]]

The shape of register `beta` is (4096, 2).
The bytes in register `beta`, shot by shot:
[[  0   0]
 [  1 248]
 [  0   0]
 ...
 [  0   0]
 [  0   0]
 [  0   0]]

The shape of `beta` after post-selection is (0, 2).
The bytes in `beta` after post-selection:
[]
The shape of `beta` after bit-wise slicing is (4096, 1).
The bytes in `beta` after bit-wise slicing:
[[0]
 [0]
 [0]
 ...
 [0]
 [0]
 [0]]

The shape of `beta` after shot-wise slicing is (5, 2).
The bytes in `beta` after shot-wise slicing:
[[  0   0]
 [  1 248]
 [  0   0]
 [  0   0]
 [  0   0]]

Exp. val. for observable `SparsePauliOp(['ZZZZZZZZZ'],
              coeffs=[1.+0.j])` is: 0.07470703125
Exp. val. for observable `SparsePauliOp(['IIIIIIIIZ'],
              coeffs=[1.+0.j])` is: 0.0244140625

The shape of the merged results is (4096, 2).
The bytes of the merged results:
[[  0   0]
 [  3 240]
 [  0   0]


عندما تحتوي دائرة على أكثر من سجل كلاسيكي واحد، يتم تخزين النتائج في كائنات `BitArray` مختلفة. يعدّل المثال التالي المقتطف السابق بتقسيم السجل الكلاسيكي إلى سجلين مستقلين:

In [6]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'execution' : {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-05-13 14:23:00', stop='2026-05-13 14:23:02', size=4096>)])},
'version' : 2,

The metadata of the PubResult result is:
'circuit_metadata' : {},


<span id="execution-spans"></span>
### View execution spans

The results of [`SamplerV2`](/docs/api/qiskit-ibm-runtime/sampler-v2) jobs executed in Qiskit Runtime contain execution timing information in their metadata.
This timing information can be used to place upper and lower timestamp bounds on when particular shots were executed on the QPU.
Shots are grouped into [`ExecutionSpan`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span) objects, each of which indicates a start time, a stop time, and a specification of which shots were collected in the span.

An execution span specifies which data was executed during its window by providing an [`ExecutionSpan.mask`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span#mask) method. This method, given any [Primitive Unified Block (PUB)](/docs/guides/primitive-input-output#pubs) index, returns a boolean mask that is `True` for all shots executed during its window. PUBs are indexed by the order in which they were given to the Sampler run call. If, for example, a PUB has shape `(2, 3)` and was run with four shots, then the mask's shape is `(2, 3, 4)`. See the [execution_span](/docs/api/qiskit-ibm-runtime/execution-span) API page for full details.

To view execution span information, review the metadata of the result returned by `SamplerV2`, which comes in the form of an `ExecutionSpans` object. This object is a list-like container containing instances of subclasses of `ExecutionSpan`, such as `SliceSpan`.

Example:

In [7]:
# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.

job = sampler.run([sampler_pub], shots=4)

result = job.result()
spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2026-05-13 14:23:20', stop='2026-05-13 14:23:21', size=24>)])


In [8]:
from qiskit.primitives import BitArray

# Get the mask of the 1st PUB for the 0th span.
mask = spans[0].mask(0)

# Decide whether the 0th shot of parameter set (1, 2) occurred in this span.
in_this_span = mask[2, 1, 0]

# Create a new bit array containing only the PUB-1 data collected during this span.
bits = result[0].data.meas
filtered_data = BitArray(bits.array[mask], bits.num_bits)

Execution spans can be filtered to include information pertaining to specific PUBs, selected by their indices:

In [9]:
# take the subset of spans that reference data in PUBs 0 or 2
spans.filter_by_pub([0, 2])

ExecutionSpans([DoubleSliceSpan(<start='2026-05-13 14:23:20', stop='2026-05-13 14:23:21', size=24>)])

## بيانات تعريف النتيجة
بالإضافة إلى نتائج التنفيذ، يحتوي كلٌّ من كائنَي `PrimitiveResult` و `SamplerPubResult` على سمة بيانات تعريف عن المهمة التي تم إرسالها. يمكن العثور على البيانات الوصفية التي تحتوي على معلومات لجميع PUBs المُرسَلة (مثل [خيارات وقت التشغيل](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options) المختلفة المتاحة) في `PrimitiveResult.metatada`، بينما توجد البيانات الوصفية الخاصة بكل PUB في `SamplerPubResult.metadata`.

تتضمن بيانات تعريف نتيجة Sampler أيضاً معلومات توقيت التنفيذ تُسمى [_نطاق التنفيذ_](#execution-spans).

> **Note:** في حقل البيانات الوصفية، يمكن لتطبيقات الأدوات البدائية إرجاع أي معلومات تتعلق بالتنفيذ ذات صلة بها، ولا توجد أزواج مفتاح-قيمة مضمونة من قِبل الأداة البدائية الأساسية. وبالتالي، قد تختلف البيانات الوصفية المُعادة في تطبيقات الأدوات البدائية المختلفة.

In [10]:
print("Number of execution spans:", len(spans))
print("  Start of the first span:", spans.start)
print("     End of the last span:", spans.stop)
print("       Total duration (s):", spans.duration)

Number of execution spans: 1
  Start of the first span: 2026-05-13 14:23:20.441518
     End of the last span: 2026-05-13 14:23:21.564845
       Total duration (s): 1.123327


Extract and inspect a particular span:

In [11]:
spans.sort()
print(" Start of first span:", spans[0].start)
print("   End of first span:", spans[0].stop)
print("#shots in first span:", spans[0].size)

 Start of first span: 2026-05-13 14:23:20.441518
   End of first span: 2026-05-13 14:23:21.564845
#shots in first span: 24


<span id="execution-spans"></span>
### عرض نطاقات التنفيذ
تحتوي نتائج مهام [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2) المنفذة في Qiskit Runtime على معلومات توقيت التنفيذ في بياناتها الوصفية.
يمكن استخدام معلومات التوقيت هذه لوضع حدود زمنية علوية وسفلية على وقت تنفيذ الـ shots المحددة على QPU.
يتم تجميع الـ shots في كائنات [`ExecutionSpan`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span)، يشير كل منها إلى وقت بدء ووقت توقف وتحديد الـ shots التي تم جمعها في هذا النطاق.

يحدد نطاق التنفيذ البيانات التي تم تنفيذها أثناء نافذته الزمنية بتوفير طريقة [`ExecutionSpan.mask`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span#mask). هذه الطريقة، عند إعطائها أي فهرس [Primitive Unified Block (PUB)](/guides/primitive-input-output#pubs)، تعيد قناعاً منطقياً يكون `True` لجميع الـ shots المنفذة خلال نافذتها الزمنية. يُفهرَس PUBs بالترتيب الذي أُعطيت به لاستدعاء تشغيل Sampler. إذا كان لـ PUB ما شكل `(2, 3)` وتم تشغيله بأربعة shots، فإن شكل القناع يكون `(2, 3, 4)`. انظر صفحة API [execution_span](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span) للتفاصيل الكاملة.

لعرض معلومات نطاق التنفيذ، راجع البيانات الوصفية للنتيجة التي يعيدها `SamplerV2`، والتي تأتي في شكل كائن `ExecutionSpans`. هذا الكائن هو حاوية شبيهة بالقائمة تحتوي على مثيلات من فئات فرعية لـ `ExecutionSpan`، مثل `SliceSpan`.

مثال: